---

# **MODEL INFERENCE**
### Customer Review Sentiment Analysis — Consumer Electronics Case Study

In this notebook, the saved pipeline (`models/best_model_sentiment.pkl`) is loaded and used to predict the sentiment of **new reviews**. The input must be in its **original / raw format** (not preprocessed). The pipeline already contains every step: text cleaning, feature creation, imputation, outlier capping, scaling, encoding, TF-IDF and the tuned Logistic Regression classifier.

**Required input columns**

| Column | Required? | Notes |
|---|---|---|
| `review_text` | Yes | Raw text: HTML, emojis, slang and URLs are handled by the pipeline |
| `product_category` | Yes | Audio, Wearables, Smart Home, Accessories (an unknown value is ignored) |
| `review_title` | Optional | Leave empty for text sources without a title (tickets, chats, social posts) |

Other columns (e.g., `review_id`, `sales_channel`) may be included. They are not used for the prediction but are useful for reporting.

> `text_utils.py` must be in the same folder as this notebook, because the pipeline uses the custom transformers defined there.

In [1]:
# 1. Import libraries
import pandas as pd
import pickle

pd.set_option('display.max_colwidth', 90)

import sklearn

# 2. Load the saved pipeline
with open('models/best_model_sentiment.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

# 3. Confirm the model is loaded (print the steps instead of the HTML diagram,
#    which can fail when the scikit-learn version differs from the training environment)
print('scikit-learn version :', sklearn.__version__)
print('Pipeline steps       :', [name for name, _ in loaded_model.steps])
print('Classes              :', list(loaded_model.classes_))

scikit-learn version : 1.9.1
Pipeline steps       : ['review_prep', 'features', 'classifier']
Classes              : ['negative', 'neutral', 'positive']

## **1. New Raw Data**

Example of new incoming data: five new marketplace reviews plus one support-ticket message that has **no title and no star rating**. The text is kept exactly as written by customers (typos, emojis, HTML, capital letters).

In [2]:
new_df = pd.DataFrame([
    {'review_id': 'RV-NEW-001', 'source': 'Amazon review', 'product_category': 'Audio',
     'review_title': 'Love it',
     'review_text': 'Sound quality blew me away!! Battery lasts for days and pairing was instant 😍😍'},
    {'review_id': 'RV-NEW-002', 'source': 'Webstore review', 'product_category': 'Smart Home',
     'review_title': 'Disappointed',
     'review_text': 'The app keeps crashing and it wont connect to my wifi.<br/>Support never replied. Requesting a refund 😡'},
    {'review_id': 'RV-NEW-003', 'source': 'Walmart review', 'product_category': 'Accessories',
     'review_title': '',
     'review_text': 'its ok. charges fine, a bit warm. price is reasonable i guess'},
    {'review_id': 'RV-NEW-004', 'source': 'Best Buy review', 'product_category': 'Wearables',
     'review_title': 'Meh',
     'review_text': 'Step tracking is roughly accurate but the strap is uncomfortable after a few hours. Not sure if I would buy again.'},
    {'review_id': 'RV-NEW-005', 'source': 'Amazon review', 'product_category': 'Audio',
     'review_title': 'Terrible',
     'review_text': 'KEEPS DISCONNECTING FROM MY PHONE AFTER THE LAST UPDATE. TOTAL WASTE OF MONEY!!!'},
    {'review_id': 'TICKET-8812', 'source': 'Support ticket', 'product_category': 'Smart Home',
     'review_title': None,
     'review_text': 'hi, camera video freezes all the time at night and setup failed 3 times. pls help'},
])

new_df

,review_id,source,product_category,review_title,review_text
0,RV-NEW-001,Amazon review,Audio,Love it,Sound quality blew me away!! Battery lasts for days and pairing was instant 😍😍
1,RV-NEW-002,Webstore review,Smart Home,Disappointed,The app keeps crashing and it wont connect to my wifi.<br/>Support never replied. Requ...
2,RV-NEW-003,Walmart review,Accessories,,"its ok. charges fine, a bit warm. price is reasonable i guess"
3,RV-NEW-004,Best Buy review,Wearables,Meh,Step tracking is roughly accurate but the strap is uncomfortable after a few hours. No...
4,RV-NEW-005,Amazon review,Audio,Terrible,KEEPS DISCONNECTING FROM MY PHONE AFTER THE LAST UPDATE. TOTAL WASTE OF MONEY!!!
5,TICKET-8812,Support ticket,Smart Home,NaN,"hi, camera video freezes all the time at night and setup failed 3 times. pls help"


## **2. Predict Sentiment**

In [3]:
# Predict the class and the probability of each class
prediction = loaded_model.predict(new_df)
probabilities = loaded_model.predict_proba(new_df)
classes = loaded_model.classes_

result = new_df[['review_id', 'source', 'review_text']].copy()
result['predicted_sentiment'] = prediction
for i, c in enumerate(classes):
    result[f'prob_{c}'] = probabilities[:, i].round(3)
result['confidence'] = probabilities.max(axis=1).round(3)

result

,review_id,source,review_text,predicted_sentiment,prob_negative,prob_neutral,prob_positive,confidence
0,RV-NEW-001,Amazon review,Sound quality blew me away!! Battery lasts for days and pairing was instant 😍😍,positive,0.031,0.039,0.929,0.929
1,RV-NEW-002,Webstore review,The app keeps crashing and it wont connect to my wifi.<br/>Support never replied. Requ...,negative,0.902,0.055,0.043,0.902
2,RV-NEW-003,Walmart review,"its ok. charges fine, a bit warm. price is reasonable i guess",neutral,0.045,0.883,0.072,0.883
3,RV-NEW-004,Best Buy review,Step tracking is roughly accurate but the strap is uncomfortable after a few hours. No...,neutral,0.254,0.406,0.340,0.406
4,RV-NEW-005,Amazon review,KEEPS DISCONNECTING FROM MY PHONE AFTER THE LAST UPDATE. TOTAL WASTE OF MONEY!!!,negative,0.812,0.042,0.146,0.812
5,TICKET-8812,Support ticket,"hi, camera video freezes all the time at night and setup failed 3 times. pls help",negative,0.398,0.357,0.246,0.398


## **3. Prioritise for the CX Team**

The negative-class probability is used to rank reviews and tickets, so that the team responds first to the customers who are most likely to be unhappy.

In [4]:
priority = (result.sort_values('prob_negative', ascending=False)
                  [['review_id', 'source', 'predicted_sentiment', 'prob_negative']]
                  .reset_index(drop=True))
priority['action'] = priority['prob_negative'].apply(
    lambda p: 'Respond within 24h' if p >= 0.6 else ('Review this week' if p >= 0.3 else 'No action needed'))
priority

,review_id,source,predicted_sentiment,prob_negative,action
0,RV-NEW-002,Webstore review,negative,0.902,Respond within 24h
1,RV-NEW-005,Amazon review,negative,0.812,Respond within 24h
2,TICKET-8812,Support ticket,negative,0.398,Review this week
3,RV-NEW-004,Best Buy review,neutral,0.254,No action needed
4,RV-NEW-003,Walmart review,neutral,0.045,No action needed
5,RV-NEW-001,Amazon review,positive,0.031,No action needed


> **Insight:**
- The model correctly reads the **raw** text, including emojis, HTML tags (`<br/>`), slang (`pls`), missing apostrophes (`wont`, `its`) and ALL-CAPS writing. No manual preprocessing was needed.
- The clearly enthusiastic review (RV-NEW-001) is classified as **positive**. The app/support/refund complaint (RV-NEW-002) and the ALL-CAPS connectivity complaint (RV-NEW-005) are classified as **negative** with high confidence.
- The hedged reviews fall into the **neutral** class: RV-NEW-003 with high confidence (0.88), and RV-NEW-004 with low confidence (0.41) because it mixes a positive point with a complaint. Low-confidence predictions like this are good candidates for a quick human check.
- The **support ticket** (TICKET-8812), which has no title and no star rating, is classified as **negative**, but with low confidence (0.40). Ticket language ("hi", "pls help") differs from review language, so the model is less certain. The model already works outside marketplace reviews, but **adding a few hundred labelled tickets** at the next retraining is recommended before full rollout on that channel.
- Sorting by `prob_negative` gives the CX team a ready-to-use **priority queue**.

---